# 00 — Download source layers (Google Earth Engine)

**Scope:** pull raw source rasters from GEE into `input_data/` at **native resolution**
(no reprojection here — `02` reprojects/resamples to the ESRI:102008 1 km grid). For now:
the updated **human modification** layer (Theobald gHM, Y2Y asset `v202606`).

**Kernel:** select **`Python (y2y-geo)`** (the project `.venv`, Python 3.12). Requires
`earthengine-api` + `geemap` (+ `geedim`, the tiled-download backend) — pinned in
`requirements.txt`.

**Auth:** the first run on a machine needs `ee.Authenticate()` once (opens a browser);
afterwards just `ee.Initialize(project=...)`.

**Note:** gHM is ~1.4 billion pixels (~5–6 GB float32), so the download is long — geedim
streams it in tiles and merges to one GeoTIFF.

Ethan runs the cells; Claude never executes them.

In [1]:
# ---- Setup --------------------------------------------------------------
import importlib

import ee
import geemap
import rasterio

import config
importlib.reload(config)  # pick up edits to config.py when re-running this cell
from config import INPUT_DIR, PROJECT_DIR

print("earthengine-api", ee.__version__, "| geemap", geemap.__version__,
      "| rasterio", rasterio.__version__)

earthengine-api 1.7.34 | geemap 0.38.3 | rasterio 1.5.0


In [2]:
# ---- Authenticate (run ONCE per machine; opens a browser) ---------------
# After the first successful run the token is cached — you can skip / comment this out.
ee.Authenticate()

True

In [3]:
# ---- Initialize the EE API ----------------------------------------------
ee.Initialize(project="y2y-climate-benefits")
print("EE initialized")

EE initialized


In [4]:
# ---- Download layers (asset -> local GeoTIFF at native res) --------------
# Add a dict here to pull another layer; each downloads in its source CRS + native scale.
GEE_LAYERS = [
    {
        "asset": "users/DavidTheobald8/Y2Y/v202606/HM_Y2Y_2024_90_60land",
        "out":   INPUT_DIR / "human_modification" / "HM_Y2Y_2024_90_60land_v202606.tif",
        "citation": "Theobald et al., gHM human modification (Y2Y asset v202606)",
    },
]

for spec in GEE_LAYERS:
    img   = ee.Image(spec["asset"])
    proj  = img.projection().getInfo()
    scale = img.projection().nominalScale().getInfo()          # native resolution (m)
    spec["out"].parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading {spec['asset']}\n  -> {spec['out'].relative_to(PROJECT_DIR)}  "
          f"(crs {proj['crs']}, native ~{scale:.1f} m)")
    geemap.download_ee_image(
        img,
        filename=str(spec["out"]),
        region=img.geometry(),                                 # the asset's own footprint
        crs=proj["crs"],
        scale=scale,                                           # native res, native CRS
    )
    print(f"  done: {spec['out'].name}")

downloading users/DavidTheobald8/Y2Y/v202606/HM_Y2Y_2024_90_60land
  -> input_data/human_modification/HM_Y2Y_2024_90_60land_v202606.tif  (crs EPSG:4326, native ~90.0 m)


/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/.venv/lib/python3.12/site-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


...ld8/Y2Y/v202606/HM_Y2Y_2024_90_60land:   0%|          |0/1365 tiles [00:00<?]

  done: HM_Y2Y_2024_90_60land_v202606.tif


/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/.venv/lib/python3.12/site-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'users/DavidTheobald8/Y2Y/v202606/HM_Y2Y_2024_90_60land'.
  return STACClient().get(self.id)


In [5]:
# ---- Verify the downloads -----------------------------------------------
for spec in GEE_LAYERS:
    with rasterio.open(spec["out"]) as ds:
        print(f"{spec['out'].name}: {ds.width} x {ds.height} | {ds.crs} | "
              f"res {ds.res[0]:.6f} | {ds.dtypes[0]} | nodata {ds.nodata}")
        print(f"  bounds {ds.bounds}")

HM_Y2Y_2024_90_60land_v202606.tif: 42609 x 32869 | EPSG:4326 | res 0.000808 | float32 | nodata -inf
  bounds BoundingBox(left=-142.14114605845919, bottom=41.278755115161374, right=-107.69246171151536, top=67.85280768151347)
